<a href="https://colab.research.google.com/github/Evokz/BP/blob/main/Teste_Estoque.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import os
from datetime import datetime

ARQUIVO_ESTOQUE = "estoque.json"
ARQUIVO_MOVIMENTACOES = "movimentacoes.json"


def carregar(caminho, padrao):
    if not os.path.exists(caminho):
        return padrao
    with open(caminho, encoding="utf-8") as arquivo:
        return json.load(arquivo)


def salvar(caminho, dados):
    with open(caminho, "w", encoding="utf-8") as arquivo:
        json.dump(dados, arquivo, ensure_ascii=False, indent=2)


def buscar_produto(dados, codigo):
    for produto in dados["estoque"]:
        if produto["codigoProduto"] == codigo:
            return produto
    return None


def proximo_id(movimentacoes):
    if not movimentacoes:
        return 1
    return max(m["id"] for m in movimentacoes) + 1


def movimentar(dados, movimentacoes, codigo, tipo, quantidade, descricao):
    produto = buscar_produto(dados, codigo)
    if produto is None:
        raise ValueError("Produto não encontrado.")
    if quantidade <= 0:
        raise ValueError("A quantidade precisa ser maior que zero.")
    if not descricao.strip():
        raise ValueError("Informe uma descrição para a movimentação.")

    if tipo == "entrada":
        produto["estoque"] += quantidade
    else:
        if quantidade > produto["estoque"]:
            raise ValueError(
                f"Estoque insuficiente. Você tem {produto['estoque']} unidades."
            )
        produto["estoque"] -= quantidade

    movimentacao = {
        "id": proximo_id(movimentacoes),
        "data": datetime.now().strftime("%d/%m/%Y %H:%M:%S"),
        "codigoProduto": codigo,
        "tipo": tipo,
        "quantidade": quantidade,
        "descricao": descricao.strip(),
        "estoqueFinal": produto["estoque"],
    }
    movimentacoes.append(movimentacao)
    return movimentacao


def ler_inteiro(pergunta):
    while True:
        try:
            return int(input(pergunta))
        except ValueError:
            print("Digite um número inteiro.")


def mostrar_estoque(dados):
    print()
    print(f"{'Código':<8}{'Produto':<30}{'Qtde':>6}")
    print("-" * 44)
    for p in dados["estoque"]:
        print(f"{p['codigoProduto']:<8}{p['descricaoProduto']:<30}{p['estoque']:>6}")
    print()


def lancar(dados, movimentacoes, tipo):
    mostrar_estoque(dados)
    codigo = ler_inteiro("Código do produto: ")
    quantidade = ler_inteiro("Quantidade: ")
    descricao = input("Descrição (ex: compra de fornecedor, venda, ajuste): ")

    try:
        mov = movimentar(dados, movimentacoes, codigo, tipo, quantidade, descricao)
    except ValueError as erro:
        print(f"\nNão deu certo: {erro}\n")
        return

    salvar(ARQUIVO_ESTOQUE, dados)
    salvar(ARQUIVO_MOVIMENTACOES, movimentacoes)

    produto = buscar_produto(dados, codigo)
    print(f"\nMovimentação nº {mov['id']} registrada ({mov['descricao']}).")
    print(f"Estoque final de {produto['descricaoProduto']}: {mov['estoqueFinal']}\n")


def main():
    dados = carregar(ARQUIVO_ESTOQUE, {"estoque": []})
    movimentacoes = carregar(ARQUIVO_MOVIMENTACOES, [])

    while True:
        print("1 - Entrada de mercadoria")
        print("2 - Saída de mercadoria")
        print("3 - Ver estoque")
        print("4 - Histórico de movimentações")
        print("0 - Sair")
        opcao = input("Escolha: ").strip()

        if opcao == "1":
            lancar(dados, movimentacoes, "entrada")
        elif opcao == "2":
            lancar(dados, movimentacoes, "saida")
        elif opcao == "3":
            mostrar_estoque(dados)
        elif opcao == "4":
            print()
            if not movimentacoes:
                print("Nenhuma movimentação ainda.")
            for m in movimentacoes:
                print(
                    f"[{m['id']}] {m['data']} | produto {m['codigoProduto']} | "
                    f"{m['tipo']} de {m['quantidade']} | {m['descricao']} | "
                    f"saldo {m['estoqueFinal']}"
                )
            print()
        elif opcao == "0":
            break
        else:
            print("Opção inválida.\n")


if __name__ == "__main__":
    main()

1 - Entrada de mercadoria
2 - Saída de mercadoria
3 - Ver estoque
4 - Histórico de movimentações
0 - Sair
Escolha: 3

Código  Produto                         Qtde
--------------------------------------------
101     Caneta Azul                      150
102     Caderno Universitário             75
103     Borracha Branca                  200
104     Lápis Preto HB                   320
105     Marcador de Texto Amarelo         90

1 - Entrada de mercadoria
2 - Saída de mercadoria
3 - Ver estoque
4 - Histórico de movimentações
0 - Sair
Escolha: 2

Código  Produto                         Qtde
--------------------------------------------
101     Caneta Azul                      150
102     Caderno Universitário             75
103     Borracha Branca                  200
104     Lápis Preto HB                   320
105     Marcador de Texto Amarelo         90

Código do produto: 1
Quantidade: 1
Descrição (ex: compra de fornecedor, venda, ajuste): das

Não deu certo: Produto não encontrado.
